[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/06_tokenizer_free_bytes_blt/06_tokenizer_free_bytes_blt.ipynb)

# Módulo 06 · Tokenizer-free I: bytes y *patches* (de CANINE y ByT5 a BLT)

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 5 h (lección ≈ 3 h + proyecto ≈ 2 h) |
| **GPU recomendada** | Ninguna para `FAST_DEV_RUN=True` (CPU, ≈ 6-10 min en total). Con `FAST_DEV_RUN=False`: T4 (≈ 3-5 unidades de Colab, estimación; el modo completo entrena modelos algo mayores, sigue siendo didáctico) |
| **Prerrequisitos** | 01 (bytes/UTF-8), 02 (BPE, bytes por token), 04 (fertilidad, coste por idioma), 05 (patologías que nacen del vocabulario) |
| **Madurez del tema** | 🟡/🔴: hay un resultado a escala (BLT, 8B) y pesos abiertos, pero ningún LLM de frontera conocido lo usa en producción (ver «Mapa de madurez») |

> **Contexto Políglota Labs.** Tras el módulo 05 tienes un informe de patologías del tokenizer: tokens sub-entrenados, cuentas de letras erróneas, «impuesto» en español. Tu CTO pregunta:
> *«¿y si el modelo leyera bytes y nos olvidáramos del vocabulario?»* Esta lección te da el mecanismo (parches por entropía), el coste real de tal decisión y un banco de pruebas diminuto para
> **comprobarlo tú mismo**, sin vender humo: lo que aquí reproduces es **didáctico y a escala diminuta**, no el resultado del paper.

## 1. Objetivos de aprendizaje

1. **Explicar** por qué un modelo a nivel de bytes es costoso (longitud de secuencia) y **por qué a escala el coste dominante es el FFN por posición, no la atención**.
2. **Comparar** CANINE, ByT5, MegaByte, SpaceByte, MambaByte y BLT por *cómo agrupan* los bytes y dónde gastan cómputo.
3. **Implementar** un modelo de entropía de bytes (tabla n-grama y transformer diminuto) y **derivar** parches con las dos reglas de BLT (umbral global y restricción monótona).
4. **Calibrar** el umbral para un tamaño medio de parche objetivo y **verificar** la propiedad de *incremental patching* (BLT sí, BPE no).
5. **Construir** un *MiniBLT* (codificador local con hash de n-gramas → atención cruzada → transformer global → atención cruzada → decodificador local) y **verificar su causalidad**.
6. **Calcular** FLOPs por byte de BPE, bytes y BLT y **medir** bits-por-byte (bpb) frente a un baseline BPE a escala diminuta.
7. **Evaluar críticamente** los resultados publicados (qué prueban y qué no) y **etiquetar** la madurez de cada técnica.

## 2. 🔁 Conexión con módulos anteriores

Responde antes de seguir (la respuesta está plegada).

<details>
<summary>1 · (Módulo 01) ¿Cuántos bytes UTF-8 ocupan «n», «ñ» y «🙂»? ¿Qué implica para un modelo que lee bytes en español?</summary>

1, 2 y 4 bytes. El español ocupa ~1.0x-1.1x bytes por carácter respecto al inglés, pero otros alfabetos pagan 2-3 bytes por carácter: un modelo de bytes ve secuencias *más largas* para esos idiomas, igual que un BPE entrenado sobre inglés los trocea en más tokens.

</details>

<details>
<summary>2 · (Módulo 02) Si el vocabulario BPE crece (de 32K a 128K), ¿qué pasa con los bytes por token y con la capa de salida?</summary>

Los tokens son más largos (menos pasos del transformer) pero la matriz de embedding/proyección de salida crece. El paper de BLT lo ilustra: Llama 2 → Llama 3 sube de 3.7 a 4.4 bytes/token con una tabla de embeddings 4x mayor (BLT §2.4). Es un compromiso *fijo* vocabulario-cómputo; BLT lo rompe porque el tamaño de parche no depende de ninguna tabla.

</details>

<details>
<summary>3 · (Módulo 05) ¿Qué patologías del tokenizer desaparecerían por construcción si el modelo viera bytes? ¿Cuáles no?</summary>

Desaparecen: tokens sub-entrenados/glitch (no hay vocabulario que quede huérfano), dependencia de espacios/mayúsculas como *ids* distintos y gran parte del problema de contar letras. No desaparecen: sesgos de datos, aritmética mal formada, ataques de Unicode (los bytes siguen ahí) ni el coste: la secuencia es más larga o hace falta parchear.

</details>

> 🧭 **Ruta núcleo vs ampliación.** La lección tiene ≈ 72 celdas (≈ 3 h; las celdas de entrenamiento tardan 1–3 min en CPU). *Núcleo (≈ 2 h), lo que usa el proyecto:* secciones 3, 4 (FLOPs por byte), 6 (teoría del parcheo por entropía), 7.1–7.2 (modelo de entropía y calibración del umbral), 8 (arquitectura BLT) y 10 (qué dice el paper). *Ampliación (≈ 1 h):* 5 (familia ByT5/CANINE/MegaByte/SpaceByte), 7.3 (parcheo incremental), 9 (MiniBLT desde cero y su entrenamiento) y 11 (librerías). Si vas justo de tiempo, ejecuta las celdas de la sección 9 sin estudiarlas y vuelve a ellas tras el proyecto.

## 3. Intuición primero: la tokenización es una heurística de compresión

Un tokenizer BPE es un **compresor sin pérdida con un diccionario fijo**: agrupa los pares de bytes más frecuentes (módulo 02). El transformer gasta **el mismo cómputo en cada token**, aunque
unos sean triviales («ción» tras «ac») y otros decisivos (la primera letra de un nombre propio). Y el modelo **nunca ve los bytes**: ve ids opacos. De ahí salen los problemas que ya medimos
en los módulos 04-05 (impuesto multilingüe, glitch tokens, «strawberry»).

**La alternativa obvia es leer bytes.** Pero la secuencia se alarga ~4-5x (módulo 02) y la atención es cuadrática… aunque el paper de BLT recuerda que *a escala* el coste dominante son las capas
feed-forward que corren **en cada posición**, no la atención. La solución de la familia moderna es **agrupar bytes en parches** (*patches*) y correr el transformer grande **solo una vez por parche**:

* **Analogía de compresión.** Un compresor aritmético asigna más bits a lo impredecible. BLT hace lo análogo con el *cómputo*: mide con un modelo de bytes pequeño **cuánta incertidumbre** (entropía) hay en el siguiente byte y abre un parche nuevo — es decir, **gasta un paso del transformer grande** — donde la incertidumbre es alta.
* **Analogía de búsqueda/IR.** Es como indexar documentos: no llamas al reranker caro para *cada* documento, solo para los candidatos difíciles.

> ### 🎨 GEMINI-IMAGEN · ID: `M06-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **El espectro de la tokenización: de bytes a parches**
>
> **Qué muestra:** Muestra la misma frase en cuatro granularidades (bytes, tokens BPE, parches fijos, parches dinámicos) y el compromiso entre control fino y número de pasos del transformer grande.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Cuatro filas con la frase «El gato duerme.» segmentada en bytes, tokens BPE, parches fijos y parches dinámicos
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, estilo infografía de paper (sans-serif legible, flechas claras, sin ilustraciones decorativas ni texto inventado). Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Formato 16:9.
> Título superior: "El espectro de la tokenización: ¿cuántos bytes por paso del transformer?".
> Composición: 4 filas horizontales apiladas, todas muestran la MISMA frase en español: "El gato duerme." (15 bytes, con el espacio escrito como ␣). A la izquierda de cada fila va una etiqueta en negrita; a la derecha, un texto gris pequeño.
> Fila 1, etiqueta "Bytes": 15 cajitas grises, una por carácter (E, l, ␣, g, a, t, o, ␣, d, u, e, r, m, e, .). Texto derecha: "1 byte por paso · secuencia larga (15 pasos)".
> Fila 2, etiqueta "Tokens BPE (vocabulario fijo)": 4 bloques naranjas contiguos con los textos "El", "␣gato", "␣duerme", ".". Texto derecha: "~4.4 bytes por token en Llama 3 (dato del paper de BLT) · ids opacos".
> Fila 3, etiqueta "Parches fijos (MegaByte, k=4)": 4 bloques azules de igual ancho con los textos "El␣g", "ato␣", "duer", "me.". Texto derecha: "tamaño fijo · corta palabras al azar".
> Fila 4, etiqueta "Parches dinámicos (BLT)": 5 bloques verdes de ancho distinto con los textos "El", "␣g", "ato", "␣d", "uerme." y encima de cada bloque una pequeña barra vertical cuya altura sugiere la entropía del primer byte (más alta en "g" y "d"). Texto derecha: "tamaño según la incertidumbre del siguiente byte (ilustrativo)".
> Una flecha vertical a la izquierda de las filas, de arriba a abajo, con la leyenda "menos pasos del transformer grande ↓" y otra en el extremo opuesto "más control sobre los bytes ↑". Nota inferior en gris: "Segmentaciones ilustrativas; no son la salida de ningún modelo real".
> Evita: ilustraciones decorativas, texto ilegible, números inventados distintos de los dados, flechas ambiguas, sombras 3D.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M06-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Cuatro filas con la frase «El gato duerme.» segmentada en bytes, tokens BPE, parches fijos y parches dinámicos](assets/M06-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> Bytes      : [E][l][␣][g][a][t][o][␣][d][u][e][r][m][e][.]            15 pasos
> BPE        : [El][␣gato][␣duerme][.]                                   4 pasos  (~4.4 B/token en Llama 3)
> Fijos k=4  : [El␣g][ato␣][duer][me.]                                   4 pasos  (corta palabras)
> Dinámicos  : [El][␣g][ato][␣d][uerme.]                                 5 pasos  (frontera donde sube la entropía; ilustrativo)
> ```

> **👀 Qué debes observar**
>
> Cada bloque es un **paso del transformer grande**. Con bytes pagas 15; con BPE/parches pagas 4-5. La diferencia entre BPE y BLT no es *cuántos* pasos sino **quién decide dónde cortar**:
> BPE lo decide una tabla estática aprendida *antes* (y el modelo no ve los bytes); BLT lo decide en línea con un modelo de entropía y el modelo **sí conserva el acceso a los bytes**.

In [ ]:
!pip install -q torch numpy matplotlib tokenizers transformers datasets

In [ ]:
import os, sys, time, math, random, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
FAST_DEV_RUN = True      # True: CPU, minutos. False: GPU (T4/L4), escala completa
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", min(4, os.cpu_count() or 1))))
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN, "| torch", torch.__version__)

import re
import matplotlib.pyplot as plt
import torch.nn as nn, torch.nn.functional as F
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, VERDE, GRIS = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280"

### 3.1 Corpus Políglota (utilidades del módulo 00)

Las tres celdas siguientes son la implementación **canónica** de `carga_corpus()` (módulo 00, autor A1). Cadena: Wikipedia ES/EN vía Hugging Face → API pública de Wikipedia → texto **sintético**;
código: CodeSearchNet → funciones **reales** de la stdlib de Python. Si imprime «SINTÉTICO», los números absolutos de esta lección **no son comparables** con datos reales (las conclusiones cualitativas sí).

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
corp = carga_corpus(FAST_DEV_RUN)
print("fuente del corpus:", corp["source"], "| docs:", len(corp["docs"]))
rng = random.Random(SEED)
docs = corp["docs"][:]; rng.shuffle(docs)
cut = int(0.9 * len(docs))
DOCS_TR, DOCS_VA = docs[:cut], docs[cut:]
MAX_CHARS = 1500 if FAST_DEV_RUN else 6000
LIMITE_BYTES = None if FAST_DEV_RUN else 3_000_000        # el modelo n-grama es Python puro: acota el modo completo

def texto(ds):
    return "\n\n".join(d["text"][:MAX_CHARS] for d in ds)

TXT_TR, TXT_VA = texto(DOCS_TR), texto(DOCS_VA)
TR = np.frombuffer(TXT_TR.encode("utf-8"), dtype=np.uint8).copy()[:LIMITE_BYTES]
VA = np.frombuffer(TXT_VA.encode("utf-8"), dtype=np.uint8).copy()[:LIMITE_BYTES]
print(f"bytes train={len(TR):,} | val={len(VA):,}")
for lang in ["es", "en", "code"]:
    t = "".join(d["text"][:MAX_CHARS] for d in DOCS_TR if d["lang"] == lang)
    print(f"{lang:>4}: {len(t):>8,} chars -> {len(t.encode()):>8,} bytes  ({len(t.encode())/max(1, len(t)):.3f} bytes/char)")

## 4. El coste de leer bytes: FLOPs por byte

Para una capa transformer con ancho $d$ y contexto $n$, el forward por posición cuesta aproximadamente

$$\text{FLOPs/posición/capa} \approx 2\,(4d^2 + 2\cdot 4d^2) + 4\,\tfrac{n}{2}\,d,$$

donde $4d^2$ son las proyecciones $Q,K,V,O$, $8d^2$ las dos matrices del FFN (con expansión 4x), y el último término es la atención causal (en media se atiende a $n/2$ posiciones). Cada MAC cuenta 2 FLOPs.
Para $d=4096$ el término de atención solo es relevante si $n$ es enorme: **el coste lo dominan las matrices que corren en cada posición** (de ahí el título del paper: *los parches escalan mejor que los tokens*).

Si el transformer grande corre una vez por **unidad** de $b$ bytes (token o parche), los FLOPs *por byte* de la parte global se dividen por $b$:

$$\text{FLOPs}_{\text{BLT}}/\text{byte} \;\approx\; \frac{L_g\,f(d_g,\,n/ps)}{ps} \;+\; (L_{enc}+L_{dec})\,f(d_l, w) \;+\; 2 d_l\cdot 256 \;+\; \text{cross}$$

con $ps$ el tamaño medio de parche, $f$ el coste por capa de arriba y $w$ la ventana de los modelos locales. (El paper da las fórmulas exactas en su §4.5 y Apéndice B; esta es una simplificación **didáctica**.)

In [ ]:
def flops_capa(d, ctx, ffn=4):
    """FLOPs (forward) de UNA capa transformer por posición: QKVO (4d^2) + FFN (2*ffn*d^2) + atención causal (~ctx/2)."""
    return 2 * (4 * d * d + 2 * ffn * d * d) + 4 * (ctx / 2) * d

def flops_por_byte(modelo, d_g=4096, L_g=32, ctx_bytes=8192, vocab_bpe=128_000, bpt=4.4, ps=4.5,
                   d_l=1024, L_enc=1, L_dec=6, ventana=512):
    """FLOPs de forward POR BYTE de texto. Config ILUSTRATIVA tipo 8B (NO es la del paper)."""
    if modelo == "bpe":      # un paso del transformer grande por token + proyección a un vocabulario grande
        return (L_g * flops_capa(d_g, ctx_bytes / bpt) + 2 * d_g * vocab_bpe) / bpt
    if modelo == "bytes":    # transformer isotrópico: un paso por byte, atención sobre todos los bytes
        return L_g * flops_capa(d_g, ctx_bytes) + 2 * d_g * 256
    if modelo == "blt":      # global (1 paso por parche) + modelos locales (1 paso por byte) + atenciones cruzadas
        glob = L_g * flops_capa(d_g, ctx_bytes / ps) / ps
        local = (L_enc + L_dec) * flops_capa(d_l, ventana) + 2 * d_l * 256
        cross = 4 * d_l * d_g
        return glob + local + cross
    raise ValueError(modelo)

base = flops_por_byte("bpe")
print(f"{'BPE (4.4 B/token, vocab 128K)':38s} {base/1e9:6.2f} GFLOPs/byte  (1.00x)")
print(f"{'Bytes isotrópico (1 B/paso)':38s} {flops_por_byte('bytes')/1e9:6.2f} GFLOPs/byte  ({flops_por_byte('bytes')/base:.2f}x)")
for ps in [4.5, 6, 8]:
    f = flops_por_byte("blt", ps=ps)
    print(f"{'BLT, patch medio %.1f B' % ps:38s} {f/1e9:6.2f} GFLOPs/byte  ({f/base:.2f}x)")

pss = np.linspace(2, 12, 60)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(pss, [flops_por_byte("blt", ps=p) / 1e9 for p in pss], color=AZUL, lw=2, label="BLT (modelo global fijo)")
ax.axhline(base / 1e9, color=NARANJA, ls="--", label="BPE (4.4 bytes/token)")
ax.axhline(flops_por_byte("bytes") / 1e9, color=GRIS, ls=":", label="Transformer sobre bytes")
ax.set_yscale("log"); ax.set_xlabel("tamaño medio de parche (bytes)"); ax.set_ylabel("GFLOPs por byte (forward)")
ax.set_title("El tamaño de parche es una perilla de cómputo (config. ilustrativa)"); ax.legend(); plt.show()

> **👀 Qué debes observar**
>
> * La curva de BLT **cae como ~1/ps** hasta que el coste de los modelos locales (que corren por byte) pone un suelo. Con la config. ilustrativa, BLT con ps≈4.5 cuesta *lo mismo* que BPE (4.4 B/token): la gracia es que **ps no está atado a un vocabulario** y puedes subirlo.
> * Un transformer isotrópico sobre bytes es ~4-5x más caro que BPE (un paso por byte). Ese es el muro que MegaByte/SpaceByte/BLT atacan.
> * ⚠️ FLOPs ≠ latencia. El propio paper admite que su implementación puede no estar a la par en tiempo de reloj con los transformers tokenizados (§9).

### 4.1 ¿Cuánto mide un «paso» en BPE? Bytes por token según vocabulario e idioma

Entrenamos BPE de varios tamaños con `tokenizers` (módulo 02-03) sobre el corpus Políglota y medimos **bytes por token**: es el «patch size» implícito del BPE.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

def entrena_bpe(docs_, vocab_size):
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    trn = trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                              show_progress=False)
    tk.train_from_iterator((d["text"][:MAX_CHARS] for d in docs_), trn)
    return tk

def bytes_por_token(tk, textos):
    n_b = sum(len(t.encode()) for t in textos)
    return n_b / sum(len(tk.encode(t).ids) for t in textos)

VOCABS = [300, 512, 1024, 2048] if FAST_DEV_RUN else [512, 4096, 16384, 32000]
BPE = {v: entrena_bpe(DOCS_TR, v) for v in VOCABS}
TK = BPE[VOCABS[1]]                                   # el BPE "de trabajo" del resto de la lección
res = {lang: [bytes_por_token(BPE[v], [d["text"][:MAX_CHARS] for d in DOCS_VA if d["lang"] == lang]) for v in VOCABS]
       for lang in ["es", "en", "code"]}

fig, ax = plt.subplots(figsize=(7.5, 4))
for lang, col in zip(["es", "en", "code"], [AZUL, NARANJA, VERDE]):
    ax.plot(VOCABS, res[lang], "o-", color=col, label=lang)
ax.axhline(1.0, color=GRIS, ls=":", label="bytes (1 B/unidad)")
ax.set_xscale("log"); ax.set_xlabel("tamaño del vocabulario BPE"); ax.set_ylabel("bytes por token")
ax.set_title("Más vocabulario = tokens más largos (el 'patch size' de BPE)"); ax.legend(); plt.show()
print({l: [round(x, 2) for x in v] for l, v in res.items()})

> **👀 Qué debes observar**
>
> * El tamaño medio de «parche» de BPE **crece con el vocabulario** (y con lo repetitivo que sea el idioma/dominio: el código suele comprimirse más que la prosa). Para pasar de ~4.4 a ~8 bytes/token tendrías que agrandar mucho el vocabulario y su tabla de embeddings.
> * Un vocabulario entrenado con poca cantidad de ES penaliza al ES frente al EN (módulo 04): en bytes no hay «impuesto de vocabulario», solo el impuesto de UTF-8.
> * (Si el corpus es sintético, las magnitudes absolutas no son las de Wikipedia/CodeSearchNet reales: mira las *tendencias*.)

## 5. La familia de modelos a nivel de byte/carácter

| Modelo | Idea | Cómo agrupa | Debilidad señalada |
|---|---|---|---|
| **CANINE** (2021) | Encoder de caracteres: convolución con *stride* + atención local → transformer profundo sobre secuencia reducida → *upsampling* | stride fijo (4) | Solo encoder (no genera); un arquitectura distinta por tarea |
| **ByT5** (2021) | T5 estándar sobre bytes con casi ningún cambio | no agrupa | Secuencias ~4x más largas: coste alto en entrenamiento e inferencia |
| **Hourglass / Dynamic Token Pooling** (2021-22) | Transformer jerárquico con *down/upsampling*; segmentación aprendida o por heurística | fija o dinámica | Escala pequeña (≤150M / 40M) |
| **MegaByte** (2023) | Decodificador multiescala: modelo *global* sobre parches + modelo *local* dentro del parche | parches **fijos** de $k$ bytes | Parches fijos cortan palabras al azar; en el análisis de BLT queda por detrás de BPE a escala |
| **SpaceByte** (2024) | Parche nuevo tras cada byte tipo espacio | por **espacios** | No varía el tamaño de parche; no gestiona bien todos los idiomas |
| **MambaByte** (2024) | SSM (Mamba) sobre bytes, sin parches | no agrupa | Escala moderada (353M–972M en el paper); coste lineal pero sin ahorro de pasos |
| **BLT** (2024) | Parches por **entropía** del siguiente byte + codificador/decodificador locales con *cross-attention* + hash de n-gramas | **dinámica**, controlable | Sin leyes de escala propias; ver «Limitaciones» más abajo |
| **H-Net** (2025) | El segmentador **se aprende** de extremo a extremo (módulo 07) | aprendida | Escala hasta ≈1.3B (FLOPs de un GPT-3 XL) en el paper |

### 5.1 Cuatro maneras de agrupar la misma frase

Código sencillo para las tres reglas «sin modelo» (stride, espacios, BPE); la regla por entropía llega en la sección 7.

In [ ]:
MUESTRA = "El servidor procesa las peticiones entrantes si falta la variable de entorno."

def inicios_stride(n, k=4):
    s = np.zeros(n, bool); s[::k] = True; return s

def es_espacio(b):                      # byte "tipo espacio": ni letra latina, ni dígito, ni continuación UTF-8
    return not (48 <= b <= 57 or 65 <= b <= 90 or 97 <= b <= 122 or 0x80 <= b <= 0xBF or b >= 0xC0)

def inicios_espacio(data):             # SpaceByte: parche nuevo tras un byte-espacio (cada parche con >=1 byte no-espacio)
    s = np.zeros(len(data), bool); s[0] = True
    for i in range(1, len(data)):
        s[i] = es_espacio(int(data[i - 1])) and not es_espacio(int(data[i]))
    return s

def inicios_bpe(tk, txt):
    off = np.cumsum([0] + [len(c.encode()) for c in txt])        # offset en caracteres -> en bytes
    s = np.zeros(off[-1], bool)
    for a, _ in tk.encode(txt).offsets:
        s[off[a]] = True
    return s

def marca(txt_bytes, starts):
    return "".join(("|" if s and i else "") + bytes([c]).decode("latin-1") for i, (c, s) in enumerate(zip(txt_bytes, starts)))

mb = MUESTRA.encode()
for nombre, st in [("stride k=4 (MegaByte)", inicios_stride(len(mb), 4)), ("espacios (SpaceByte)", inicios_espacio(mb)),
                   (f"BPE vocab={VOCABS[1]}", inicios_bpe(TK, MUESTRA))]:
    print(f"{nombre:24s} {st.sum():2d} parches | {marca(mb, st)}")

> **👀 Qué debes observar**
>
> * **Stride** corta en cualquier sitio (`El s|erv|idor…`): una misma palabra se parte de formas distintas según su posición — el modelo debe aprender todas.
> * **Espacios** y **BPE** respetan palabras, pero BPE usa una tabla estática y SpaceByte *siempre* abre parche tras un espacio (no puede hacer parches más largos en texto predecible, ni más cortos dentro de una palabra difícil).

## 6. Teoría: parches por entropía (BLT)

**Modelo de entropía.** Se entrena un modelo de lenguaje de bytes *pequeño* $p_e$ en los mismos datos. Para cada posición $i$ se calcula la incertidumbre sobre el *siguiente* byte dado el pasado:

$$H(x_i) \;=\; -\sum_{v\in\mathcal{V}} p_e(x_i = v \mid \boldsymbol{x}_{<i})\,\log p_e(x_i = v \mid \boldsymbol{x}_{<i}),\qquad \mathcal{V}=\{0,\dots,255\}$$

($\mathcal{V}$ es el alfabeto de bytes; $H$ se mide en bits si el logaritmo es base 2.) En el paper, el modelo de entropía de los experimentos principales es un transformer de **100M parámetros, 14 capas, ancho 512 y ventana deslizante de 512 bytes** (§4.2), y notan que si su campo receptivo es pequeño puede **codificarse como tabla de búsqueda** — justo lo que hace nuestra tabla n-grama.

**Dos reglas para abrir un parche en $i$** (BLT §2.3):

$$\text{Umbral global: } H(x_i) > \theta_g \qquad\qquad \text{Restricción monótona aproximada: } H(x_i) - H(x_{i-1}) > \theta_r$$

La segunda detecta *rupturas* de una entropía que debería decrecer dentro de un parche (tras la primera letra de una palabra, el resto es más fácil) y sufre menos *entropy drift* cuando el contexto cambia (§4.4).

**Ejemplo a mano** (con $\log_2$, en bits). Tras «el servid» el modelo de entropía casi sabe lo que viene: $p(\texttt{o})=0.9$ y reparte 0.1 entre otros 10 bytes ($0.01$ cada uno):
$H=-0.9\log_2 0.9-10\cdot0.01\log_2 0.01=0.137+0.664=\mathbf{0.80}$ bits. Tras «el␣» (inicio de palabra) supón 16 iniciales igual de probables: $H=\log_2 16=\mathbf{4}$ bits. Con $\theta_g=2$ bits se abre parche en el inicio de palabra y **no** dentro de «servidor». Es la misma idea que un compresor aritmético (módulo 02, §9.5): lo predecible cuesta pocos bits y se agrupa; lo sorprendente cuesta muchos y recibe su propio paso de cómputo.

**Propiedad clave: parcheo incremental.** Al generar no existen los bytes futuros, así que la función de parcheo debe cumplir $f_p(\boldsymbol{x}_{<i}) = f_p(\boldsymbol{x})_{<i}$. Como $H(x_i)$ solo depende de $\boldsymbol{x}_{<i}$, el parcheo por entropía la cumple. **BPE no**: el mismo prefijo se tokeniza distinto según lo que venga después.

**Bits por byte (bpb).** Para comparar modelos con distinta segmentación no sirve la perplejidad por token:

$$\mathrm{BPB}(\boldsymbol{x}) \;=\; \frac{\mathcal{L}_{CE}(\boldsymbol{x})}{\ln 2\cdot n_{\text{bytes}}}$$

con $\mathcal{L}_{CE}$ la entropía cruzada total (en nats) de la secuencia y $n_{\text{bytes}}$ su longitud en bytes. (Detalle fino: para modelos tokenizados es una *aproximación*, porque asume que la probabilidad de la cadena es la de su tokenización canónica — el paper de H-Net lo advierte.)

### 6.1 Un modelo de entropía como tabla de búsqueda (n-grama) y como transformer

Primero la **tabla n-grama** con suavizado Witten-Bell (conteos de bytes, orden $n$): no es una red neuronal, pero es exactamente un «modelo de bytes con campo receptivo pequeño».

In [ ]:
from collections import defaultdict

class NgramByteLM:
    """Modelo de bytes de orden n con suavizado Witten-Bell: una TABLA de conteos (el 'lookup table' que
    menciona el paper de BLT cuando el campo receptivo del modelo de entropía es pequeño)."""
    def __init__(self, orden=4):
        self.orden = orden
        self.cuentas = [defaultdict(lambda: np.zeros(256, np.int32)) for _ in range(orden + 1)]
    def fit(self, data):
        b = data.tobytes()
        for n in range(self.orden + 1):
            for i in range(n, len(b)):
                self.cuentas[n][b[i - n:i]][b[i]] += 1
        return self
    def probs(self, ctx: bytes):
        p = np.full(256, 1 / 256)
        for n in range(self.orden + 1):
            if n > len(ctx):
                break
            c = self.cuentas[n].get(ctx[len(ctx) - n:] if n else b"")
            if c is None:
                break
            N, T = c.sum(), (c > 0).sum()
            p = (c + T * p) / (N + T)                      # interpolación Witten-Bell con el orden inferior
        return p
    def entropias(self, data):
        """H[i] = entropía (bits) de la distribución del byte i dado x_<i. H[0] = 8 (sin contexto)."""
        b = data.tobytes(); H = np.zeros(len(b), np.float32)
        for i in range(len(b)):
            p = self.probs(b[max(0, i - self.orden):i]); H[i] = -(p * np.log2(p + 1e-12)).sum()
        return H
    def bpb(self, data):
        b = data.tobytes(); tot = 0.0
        for i in range(len(b)):
            tot -= np.log2(self.probs(b[max(0, i - self.orden):i])[b[i]])
        return tot / len(b)

t0 = time.time()
NG = {n: NgramByteLM(n).fit(TR) for n in [0, 1, 2, 3, 4, 5]}
VAL_N = VA[:6000]
bpb_ng = {n: NG[n].bpb(VAL_N) for n in NG}
print(f"ajuste + evaluación n-gramas: {time.time()-t0:.1f}s")
print({f"orden {n}": round(float(v), 3) for n, v in bpb_ng.items()}, "(bits por byte, validación)")

Ahora el **transformer diminuto** de bytes (el análogo, en pequeño, de los 100M parámetros del paper):

In [ ]:
class TinyGPT(nn.Module):
    """Transformer causal pequeño de vocabulario arbitrario (bytes: 256; tokens BPE: |V|)."""
    def __init__(self, vocab, d=64, L=2, ctx=128, H=4):
        super().__init__()
        self.emb, self.pos = nn.Embedding(vocab, d), nn.Embedding(ctx, d)
        layer = nn.TransformerEncoderLayer(d, H, 4 * d, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
        self.tr = nn.TransformerEncoder(layer, L, enable_nested_tensor=False)
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, vocab, bias=False)
    def forward(self, x):
        T = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(T, device=x.device))
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1)
        return self.head(self.ln(self.tr(h, mask=mask, is_causal=True)))

def lote(data, B, T, g):
    ix = torch.randint(0, len(data) - T - 1, (B,), generator=g)
    x = torch.stack([torch.from_numpy(data[i:i + T].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + T + 1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

def entrena_bytelm(d=64, L=2, ctx=128, pasos=300, B=32, lr=3e-3):
    m = TinyGPT(256, d, L, ctx).to(device); opt = torch.optim.AdamW(m.parameters(), lr=lr)
    g = torch.Generator().manual_seed(SEED)
    for paso in range(pasos):
        x, y = lote(TR, B, ctx, g)
        loss = F.cross_entropy(m(x).reshape(-1, 256), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()

@torch.no_grad()
def entropias_nn(m, data, ctx=128, stride=64):
    """Entropía de cada byte con ventanas deslizantes; cada byte usa el contexto más largo disponible."""
    n = len(data); H = np.zeros(n, np.float32); H[0] = 8.0; hecho = np.zeros(n, bool); hecho[0] = True
    starts = list(range(0, n - ctx + 1, stride)) + [n - ctx]
    for s in starts:
        x = torch.from_numpy(data[s:s + ctx].astype(np.int64))[None].to(device)
        p = F.softmax(m(x)[0], -1); h = -(p * torch.log2(p + 1e-12)).sum(-1).cpu().numpy()   # h[t] -> byte s+t+1
        idx = np.arange(1, ctx) + s
        nuevo = ~hecho[idx]; H[idx[nuevo]] = h[:-1][nuevo]; hecho[idx[nuevo]] = True
    return H

t0 = time.time()
PASOS_LM = 600 if FAST_DEV_RUN else 3000
LM = entrena_bytelm(d=96 if FAST_DEV_RUN else 256, L=2 if FAST_DEV_RUN else 6, pasos=PASOS_LM)
H_va_nn = entropias_nn(LM, VA[:6000])
with torch.no_grad():
    g = torch.Generator().manual_seed(1); x, y = lote(VA, 64, 128, g)
    bpb_nn = F.cross_entropy(LM(x).reshape(-1, 256), y.reshape(-1)).item() / math.log(2)
print(f"transformer de bytes ({sum(p.numel() for p in LM.parameters())/1e6:.2f}M params, {PASOS_LM} pasos, {time.time()-t0:.0f}s): "
      f"{bpb_nn:.3f} bpb val | n-grama orden 4: {bpb_ng[4]:.3f} | orden 0: {bpb_ng[0]:.3f}")

> **👀 Qué debes observar**
>
> * Con pocos datos, la **tabla n-grama de orden 4-5 suele ganar** al transformer diminuto (memoriza); con más datos y entrenamiento el transformer mejora a n-gramas. Para *decidir dónde cortar* nos basta cualquiera de los dos si captura «aquí empieza una palabra nueva».
> * El paper de BLT ablaciona el modelo de entropía y ve **rendimientos decrecientes por encima de ~50M parámetros y 512 bytes de contexto** (§7): el modelo de patching **no necesita ser grande**.

## 7. Parches por entropía en acción

### 7.1 Calibrar el umbral para un tamaño medio de parche

In [ ]:
def inicios_global(H, theta):                    # BLT: nuevo parche si H(x_i) > theta
    s = H > theta; s[0] = True; return s

def inicios_monotono(H, theta_r):                # BLT: nuevo parche si H(x_i) - H(x_{i-1}) > theta_r
    d = np.empty_like(H); d[0] = np.inf; d[1:] = H[1:] - H[:-1]
    s = d > theta_r; s[0] = True; return s

def calibra(H, objetivo, fn, lo=-6.0, hi=9.0):
    """Bisección del umbral para que el tamaño medio de parche (bytes/parche) sea `objetivo`."""
    for _ in range(40):
        mid = (lo + hi) / 2
        if len(H) / fn(H, mid).sum() > objetivo:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2

def longitudes(starts):
    idx = np.flatnonzero(starts); return np.diff(np.r_[idx, len(starts)])

H_TR = NG[4].entropias(TR); H_VA = NG[4].entropias(VA)             # tabla n-grama (orden 4) como modelo de entropía
OBJ = 4.5                                                            # patch medio de BLT-Entropy en su mezcla de entrenamiento
TH_G = calibra(H_TR, OBJ, inicios_global); TH_M = calibra(H_TR, OBJ, inicios_monotono)
for nombre, fn, th in [("global", inicios_global, TH_G), ("monótono", inicios_monotono, TH_M)]:
    print(f"umbral {nombre:9s} θ={th:6.3f} -> tamaño medio val = {len(H_VA)/fn(H_VA, th).sum():.2f} bytes/parche")

### 7.2 La entropía del siguiente byte (réplica en pequeño de la Figura 4 del paper)

In [ ]:
def traza(ax, snippet, titulo, theta):
    b = snippet.encode()[:90]
    H = entropias_nn(LM, np.frombuffer(snippet.encode(), np.uint8).copy()[:90], ctx=min(128, len(b)), stride=16) \
        if len(b) >= 128 else NG[4].entropias(np.frombuffer(b, np.uint8))
    st = inicios_global(H, theta)
    ax.bar(range(len(b)), H, color=[NARANJA if s else AZUL for s in st], width=.85)
    ax.axhline(theta, color="crimson", ls="--", lw=1, label=f"umbral θ={theta:.2f}")
    for i in np.flatnonzero(st)[1:]:
        ax.axvline(i - .5, color=GRIS, lw=.6)
    ax.set_xticks(range(len(b))); ax.set_xticklabels([("␣" if c == 32 else chr(c) if 32 < c < 127 else "·") for c in b], fontsize=6)
    ax.set_title(titulo, fontsize=9, loc="left"); ax.set_ylabel("H (bits)"); ax.legend(loc="upper right", fontsize=7)

ejemplos = {}
for lang in ["es", "en", "code"]:
    d = next(d for d in DOCS_VA if d["lang"] == lang)
    ejemplos[lang] = d["text"].strip().replace("\n", " ")[:90]
fig, axs = plt.subplots(3, 1, figsize=(11, 7.5))
for ax, (lang, snip) in zip(axs, ejemplos.items()):
    traza(ax, snip, f"{lang}: entropía del siguiente byte (naranja = inicio de parche)", TH_G)
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * La entropía **se dispara en el primer byte de una palabra** (qué viene después de un espacio) y **cae dentro de la palabra** cuando ya conoces el comienzo. Las barras naranjas son inicios de parche: tienden a coincidir con inicios de palabra, **sin que nadie le dijera dónde están los espacios**.
> * En código aparecen picos en identificadores y literales, y parches largos en estructuras repetidas. En texto sintético con plantillas, las palabras de la plantilla son casi deterministas (parches largos) y las variables, impredecibles.

In [ ]:
snip = ejemplos["es"] if len(ejemplos["es"]) > 40 else MUESTRA
sb = np.frombuffer(snip.encode(), np.uint8)
Hs = NG[4].entropias(sb)
filas = [("stride k=4", inicios_stride(len(sb), 4)), ("espacios", inicios_espacio(sb)),
         (f"BPE {VOCABS[1]}", inicios_bpe(TK, snip)), ("entropía (global)", inicios_global(Hs, TH_G))]
fig, ax = plt.subplots(figsize=(11, 2.8))
for r, (nom, st) in enumerate(filas):
    idx = np.flatnonzero(st); fin = np.r_[idx[1:], len(sb)]
    for k, (a, b_) in enumerate(zip(idx, fin)):
        ax.barh(r, b_ - a - .08, left=a, color=[AZUL, NARANJA, VERDE][k % 3], height=.7, alpha=.85)
ax.set_yticks(range(len(filas))); ax.set_yticklabels([f for f, _ in filas]); ax.invert_yaxis()
ax.set_xticks(range(len(sb))); ax.set_xticklabels([("␣" if c == 32 else chr(c) if 32 < c < 127 else "·") for c in sb], fontsize=6)
ax.set_title("La misma frase agrupada de cuatro maneras (cada bloque = un paso del transformer grande)", fontsize=10, loc="left")
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * Cada fila es una regla de segmentación y **cada bloque, un paso del transformer grande** (lo que se paga en FLOPs). Cuenta bloques: *stride* corta a ciegas (bloques iguales que parten palabras), *espacios* y *BPE* siguen las palabras, y *entropía* corta donde el siguiente byte es difícil de predecir.
> * Fíjate en que la fila de entropía tiende a coincidir con inicios de palabra **sin saber qué es un espacio**, pero también agrupa palabras muy predecibles en un solo bloque y parte las raras en varios: el cómputo se reparte según la dificultad, no según la ortografía.

In [ ]:
TKb = TK
def long_bpe(data_docs):
    return np.concatenate([[len(t) for t in TKb.encode(d["text"][:MAX_CHARS]).tokens] for d in data_docs])

L_stride = longitudes(inicios_stride(len(VA), 4)); L_space = longitudes(inicios_espacio(VA))
L_bpe = long_bpe(DOCS_VA)
L_glob = longitudes(inicios_global(H_VA, TH_G)); L_mono = longitudes(inicios_monotono(H_VA, TH_M))

grid = np.linspace(0.2, 4.0, 40)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(grid, [min(30, len(H_TR)/inicios_global(H_TR, t).sum()) for t in grid], color=AZUL, lw=2, label="umbral global")
axs[0].plot(grid, [min(30, len(H_TR)/inicios_monotono(H_TR, t-3).sum()) for t in grid], color=NARANJA, lw=2, label="restricción monótona (θ_r = θ-3)")
axs[0].axhline(OBJ, color=GRIS, ls=":"); axs[0].set_xlabel("umbral θ (bits)"); axs[0].set_ylabel("tamaño medio de parche (bytes)")
axs[0].set_title("Un solo número controla el coste: θ"); axs[0].legend()
bins = np.arange(0.5, 21.5, 1)
for L_, nom, col in [(L_stride, "stride 4", GRIS), (L_space, "espacios", VERDE), (L_bpe, f"BPE {VOCABS[1]}", NARANJA), (L_glob, "entropía (global)", AZUL)]:
    axs[1].hist(np.clip(L_, 1, 20), bins=bins, histtype="step", lw=2, density=True, label=f"{nom} (media {L_.mean():.1f})", color=col)
axs[1].set_xlabel("bytes por parche/token"); axs[1].set_ylabel("densidad"); axs[1].set_title("Distribución de longitudes (val)"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("desv. típica de la longitud -> stride:", L_stride.std().round(2), "| BPE:", L_bpe.std().round(2), "| entropía:", L_glob.std().round(2))

In [ ]:
# ¿Coinciden las fronteras de los dos modelos de entropía (tabla n-grama vs transformer)? F1 sobre una muestra de validación
N_AC = 6000
Hn, Hg = H_va_nn[:N_AC], H_VA[:N_AC]
a = inicios_global(Hn, calibra(Hn, OBJ, inicios_global)); b = inicios_global(Hg, calibra(Hg, OBJ, inicios_global))
tp = (a & b).sum(); prec, rec = tp / a.sum(), tp / b.sum()
print(f"acuerdo de fronteras tabla n-grama vs transformer: precisión={prec:.2f} recall={rec:.2f} F1={2*prec*rec/(prec+rec):.2f}")
print(f"(azar: dos segmentaciones independientes con ps={OBJ} coincidirían en ~{100/OBJ:.0f}% de las fronteras)")

> **👀 Qué debes observar**
>
> * **Izquierda:** una sola perilla ($\theta$) controla el tamaño medio de parche, y por tanto el coste. Calibramos $\theta$ para el objetivo 4.5 B/parche (el valor de BLT-Entropy en su mezcla de entrenamiento).
> * **Derecha:** stride tiene *una sola* longitud (barra); BPE y entropía tienen **distribuciones anchas** — la entropía puede dar parches largos en lo predecible y de 1 byte en lo difícil. Con $\theta$ calibrado las medias coinciden: la diferencia es *dónde* se corta.

In [ ]:
def por_idioma(lang):
    Hs_ = [NG[4].entropias(np.frombuffer(d["text"][:MAX_CHARS].encode(), np.uint8)) for d in DOCS_VA if d["lang"] == lang]
    H_ = np.concatenate(Hs_)
    ps_ = np.mean([len(h) / inicios_global(h, TH_G).sum() for h in Hs_])
    return H_.mean(), ps_

stats = {lang: por_idioma(lang) for lang in ["es", "en", "code"]}
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
langs = list(stats)
axs[0].bar(langs, [stats[l][0] for l in langs], color=[AZUL, NARANJA, VERDE]); axs[0].set_title("Entropía media por byte (bits)")
axs[1].bar(langs, [stats[l][1] for l in langs], color=[AZUL, NARANJA, VERDE]); axs[1].set_title(f"Tamaño medio de parche con θ={TH_G:.2f} fijo")
plt.tight_layout(); plt.show()
print({l: (round(float(a), 2), round(float(b), 2)) for l, (a, b) in stats.items()})

> **👀 Qué debes observar**
>
> Con **el mismo $\theta$** para todos, los idiomas/dominios con menor entropía por byte obtienen parches más largos (menos cómputo por byte) y los más impredecibles obtienen parches más cortos. Así el cómputo **sigue a la densidad de información** — y por eso, en teoría, un modelo de bytes con parches dinámicos no castiga a un idioma solo por su vocabulario (matiz: sí lo penaliza si su *entropía* intrínseca es mayor o está poco representado en el modelo de entropía).

### 7.3 La propiedad de parcheo incremental

In [ ]:
def prueba_incremental(fn_inicios, items, n_pruebas=300, seed=SEED):
    """fp(x_<i) == fp(x)_<i ? Devuelve la fracción de cortes en los que se cumple."""
    r = random.Random(seed); ok = 0
    for _ in range(n_pruebas):
        it = r.choice(items); n = len(it if isinstance(it, bytes) else it.encode())
        i = r.randrange(5, n - 1)
        full, pref = fn_inicios(it), fn_inicios(it[:i] if isinstance(it, bytes) else it.encode()[:i].decode("utf-8", "ignore"))
        ok += int(np.array_equal(full[:len(pref)], pref))
    return ok / n_pruebas

textos_prueba = [d["text"][:300] for d in DOCS_VA[:60] if d["lang"] != "code"]
bytes_prueba = [t.encode() for t in textos_prueba]
f_ent = lambda b: inicios_global(NG[4].entropias(np.frombuffer(b, np.uint8)), TH_G)
f_bpe = lambda t: inicios_bpe(TK, t)
print(f"parches por entropía : incremental en {prueba_incremental(f_ent, bytes_prueba)*100:5.1f}% de los cortes")
print(f"tokens BPE           : incremental en {prueba_incremental(f_bpe, textos_prueba)*100:5.1f}% de los cortes")
pal = [w for w in re.findall(r"[a-záéíóúñ]{8,}", " ".join(textos_prueba))]
for w in dict.fromkeys(pal):
    full = TK.encode(" " + w).tokens
    for k in range(3, len(w) - 2):
        pref = TK.encode(" " + w[:k]).tokens
        if pref != full[:len(pref)]:
            print(f"palabra completa  ' {w}' -> {full}")
            print(f"solo el prefijo   ' {w[:k]}' -> {pref}   <- NO es un prefijo de la tokenización completa")
            break
    else:
        continue
    break

> **👀 Qué debes observar**
>
> El parcheo por entropía es incremental por construcción (100%). BPE **no lo es**: si cortas a mitad de palabra, la tokenización del prefijo difiere de la de la palabra completa, así que un generador que decidiera «¿abro token?» mirando solo el pasado no coincidiría con el tokenizer entrenado. Esto es lo que hace que BPE use un tokenizer *aparte* y BLT pueda decidir los parches *durante la generación*.

## 8. Arquitectura BLT

BLT tiene **tres** transformers (Figura 2 del paper):

1. **Codificador local** $\mathcal{E}$ (ligero, $l_E \ll l_G$ capas): embebe bytes ($256\times h_E$), les **suma embeddings hash de n-gramas** (n = 3…8) y aplica capas alternas de transformer local (atención con ventana causal) y **atención cruzada** que **agrupa bytes en parches**: las *consultas* son un *pooling* de los bytes del parche, las *claves/valores* son los bytes de ese parche (máscara por parche).
2. **Transformer latente global** $\mathcal{G}$ (grande, el que consume casi todos los FLOPs): autoregresivo sobre **parches**, con máscara causal por bloques.
3. **Decodificador local** $\mathcal{D}$ (ligero): atención cruzada **inversa** (consultas = bytes, claves/valores = parches) seguida de capas locales, que predicen los bytes del *siguiente* parche.

**Embeddings hash de n-gramas** (BLT §3.2.1). Para cada posición $i$ y cada $n\in\{3,\dots,8\}$ se hashea el n-grama $g_{i,n}=\{b_{i-n+1},\dots,b_i\}$ a una tabla de tamaño fijo y se suma:

$$e_i = x_i + \sum_{n=3}^{8} E^{\text{hash}}_n\!\big(\text{Hash}(g_{i,n})\big),\qquad \text{Hash}(g)=\text{RollPolyHash}(g)\bmod |E^{\text{hash}}_n|$$

Es la forma de dar al codificador *pseudo-vocabulario sin vocabulario*: n-gramas frecuentes obtienen embeddings propios sin tabla de tokens.

**Lo que el paper encuentra** (a escala 1B, 100B bytes; Tablas 7-9): la atención cruzada del **decodificador** es la más útil; los embeddings hash de n-gramas dan «grandes mejoras» en bpb (p. ej. de 0.850 a 0.826 en la distribución de entrenamiento al añadir n-gramas 3-8 con tablas de 400K); y, con hash, un codificador de **una capa** basta y conviene dar más capas al decodificador.

> ### 🎨 GEMINI-IMAGEN · ID: `M06-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Arquitectura del Byte Latent Transformer**
>
> **Qué muestra:** Los tres módulos de BLT (codificador local, transformer latente global, decodificador local), el agrupamiento por entropía y las atenciones cruzadas.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Diagrama de BLT: bytes con modelo de entropía → codificador local → parches → transformer global → decodificador local → bytes
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio de arquitectura, fondo blanco, estilo figura de paper (sans-serif legible, flechas claras, sin ilustraciones decorativas ni texto inventado). Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Formato 16:9.
> Título: "Byte Latent Transformer (BLT): tres módulos".
> Composición, de IZQUIERDA a DERECHA, con una fila de bytes en la parte inferior y una fila de bytes en la parte superior:
> 1) Parte inferior izquierda: una fila de 12 cajitas grises (bytes) con los textos "B","e","t","t","e","r","␣","t","h","a","n","␣". Debajo, una barra pequeña "Modelo de entropía (pequeño, 100M)" en gris que apunta con una flecha punteada a las cajitas y produce la etiqueta "entropía del siguiente byte H(x_i)". Marca con líneas verticales finas verdes tres fronteras que dividen los 12 bytes en 4 parches de tamaño distinto (3, 2, 4 y 3 bytes) con la etiqueta "fronteras: H(x_i) > θ".
> 2) Bloque azul claro, etiqueta "Codificador local (ligero)": recibe los bytes; encima un bloque pequeño etiquetado "+ embeddings hash de n-gramas (n = 3…8)". Dentro, texto "capas de transformer local + atención cruzada".
> 3) Flecha ancha etiquetada "atención cruzada: bytes → parches (consulta = parche, clave/valor = bytes del parche)" que lleva a 4 círculos naranjas etiquetados "patch 1", "patch 2", "patch 3", "patch 4".
> 4) Bloque naranja grande en el centro, etiqueta "Transformer latente global (grande)", subetiqueta "se ejecuta UNA vez por parche". Recibe los 4 círculos y emite 4 círculos de salida.
> 5) Flecha ancha etiquetada "atención cruzada: parches → bytes (consulta = byte, clave/valor = parche)" que lleva a un bloque verde claro, etiqueta "Decodificador local (ligero)".
> 6) Parte superior derecha: fila de 12 cajitas grises (bytes de salida) con el texto "predicción del siguiente byte".
> Una llave bajo el bloque global con la leyenda "la mayor parte de los FLOPs"; otra bajo los bloques locales: "corren por byte, pero son pequeños".
> Evita: ilustraciones decorativas, texto ilegible, números no indicados, flechas ambiguas, más de tres módulos principales, logotipos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M06-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Diagrama de BLT: bytes con modelo de entropía → codificador local → parches → transformer global → decodificador local → bytes](assets/M06-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> bytes  B e t t e r _ t h a n _
>         |  modelo de entropía H(x_i) -> fronteras |  |  |
>         v
>  [Codificador local + hash n-gramas + cross-attn]   (bytes -> parches)
>         v
>    patch1   patch2   patch3   patch4
>         v
>  [ Transformer latente global ]    (1 paso por parche; casi todos los FLOPs)
>         v
>  [Decodificador local + cross-attn]   (parches -> bytes)
>         v
> bytes  predicción del siguiente byte
> ```

> **👀 Qué debes observar**
>
> Localiza dónde ocurre cada cosa: el **modelo de entropía** corre aparte (preprocesado) y solo decide fronteras; la **atención cruzada** hace el cambio de resolución (bytes↔parches); el **bloque grande** solo ve parches. Si cambias $\theta$ cambias el número de parches y, con él, el coste del bloque naranja — sin tocar el modelo.

> ### 🎨 GEMINI-IMAGEN · ID: `M06-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Cómputo uniforme por token vs. cómputo adaptativo por parche**
>
> **Qué muestra:** Contrasta un transformer tokenizado (mismo gasto en cada token) con BLT (gasto concentrado donde sube la entropía).
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Dos paneles: arriba cómputo igual por token BPE; abajo cómputo variable por parche según la curva de entropía
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, sans-serif legible, sin ilustraciones decorativas ni texto inventado. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Formato 16:9.
> Título: "¿Dónde se gasta el cómputo?".
> Dos paneles horizontales apilados que comparten el mismo eje x: la frase "Daenerys Targaryen is in Game of Thrones" escrita en 41 casillas pequeñas (una por carácter, espacios como ␣).
> Panel superior, título "Transformer con tokens BPE": encima de la frase, bloques naranjas CONTIGUOS de ancho similar con los textos "Dae", "ner", "ys", "␣Tar", "gar", "yen", "␣is", "␣in", "␣Game", "␣of", "␣Thrones"; cada bloque sostiene una columna naranja de la MISMA altura. Leyenda a la derecha: "el mismo cómputo en cada token".
> Panel inferior, título "BLT con parches por entropía": una curva azul suave de entropía del siguiente byte sobre la frase, con picos al principio de cada palabra y valles dentro de ellas, y una línea roja horizontal discontinua etiquetada "umbral θ". Debajo, bloques verdes de ANCHO VARIABLE que empiezan donde la curva supera θ: columnas verdes altas y estrechas en los picos, columnas anchas y bajas donde la curva es baja. Leyenda a la derecha: "más cómputo donde hay más incertidumbre".
> Nota inferior en gris: "Ilustración conceptual; los valores de entropía no son medidas reales".
> Evita: valores numéricos de entropía inventados, adornos, texto ilegible, flechas ambiguas, sombras 3D.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M06-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Dos paneles: arriba cómputo igual por token BPE; abajo cómputo variable por parche según la curva de entropía](assets/M06-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> BPE :  [Dae][ner][ys][ Tar][gar][yen][ is][ in][ Game][ of][ Thrones]
>         ####  ####  ###  ####  ####  ####  ###  ###  #####  ###  ########   <- cómputo uniforme por token
> BLT :  entropía:  /\   /\ ...  (picos al inicio de palabra, valles dentro)
>         parches :  [Dae ner ys][ T][argaryen][ is][ in][ G][ame of][ T][hrones]   <- ancho/cómputo según θ (ilustrativo)
> ```

> **👀 Qué debes observar**
>
> El ahorro de BLT no viene de «pensar menos» sino de **no repetir cómputo caro** en bytes triviales (las terminaciones de palabra) y de poder **gastar más** en los pocos bytes difíciles. En BPE ese reparto lo fija el diccionario, no el contenido.

## 9. Implementación desde cero: *MiniBLT*

Implementamos las piezas de la figura en ~80 líneas de PyTorch. **Qué es y qué no es:**

| Sí lo hace | No lo hace (simplificaciones declaradas) |
|---|---|
| bytes → parches con la entropía de nuestra tabla n-grama; cross-attention con máscara por parche; transformer global **causal sobre parches**; decodificación de bytes con cross-attention al parche *anterior* | hash de n-gramas con un *rolling hash* simplificado y tablas diminutas (n=3-5, 4096 entradas); sin `FlexAttention` ni kernels: usamos máscaras densas; modelos de ≈1-2M parámetros; contexto de 128 bytes |

**Causalidad.** El byte $t$ se decodifica con (i) los bytes $\le t$ vía el codificador local causal y (ii) la salida del parche **anterior** $o_{j-1}$ del transformer global (que solo vio parches completos). Lo comprobamos con una prueba de «cambio un byte futuro y las salidas pasadas no varían».

In [ ]:
def mascara_causal(T, dev):
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=dev), 1)

def capas(d, L, H):
    layer = nn.TransformerEncoderLayer(d, H, 4 * d, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
    return nn.TransformerEncoder(layer, L, enable_nested_tensor=False)

class MiniBLT(nn.Module):
    """BLT en miniatura: codificador local (bytes + hash de n-gramas) -> atención cruzada que agrupa bytes en parches
    -> transformer latente global (sobre parches) -> atención cruzada que vuelve a bytes -> decodificador local."""
    def __init__(self, dl=64, dg=128, Lg=3, Le=1, Ld=2, H=4, ctx=128, ngramas=(3, 4, 5), hsize=4096):
        super().__init__()
        self.dl, self.dg, self.H, self.ngramas, self.hsize = dl, dg, H, ngramas, hsize
        self.emb = nn.Embedding(256, dl); self.hemb = nn.ModuleList([nn.Embedding(hsize, dl) for _ in ngramas])
        self.pos = nn.Embedding(ctx, dl); self.enc = capas(dl, Le, H)
        self.q_e, self.k_e, self.v_e = nn.Linear(dl, dg), nn.Linear(dl, dg), nn.Linear(dl, dg)
        self.o_e, self.pool_e, self.ppos = nn.Linear(dg, dg), nn.Linear(dl, dg), nn.Embedding(ctx, dg)
        self.glob = capas(dg, Lg, H)
        self.bos = nn.Parameter(torch.zeros(1, 1, dg)); self.kv_d = nn.Linear(dg, 2 * dl)
        self.q_d, self.o_d = nn.Linear(dl, dl), nn.Linear(dl, dl)
        self.dec = capas(dl, Ld, H); self.ln, self.head = nn.LayerNorm(dl), nn.Linear(dl, 256, bias=False)

    def hash_ngramas(self, x):
        """Embeddings hash de n-gramas de bytes (rolling hash simplificado) promediados con el embedding del byte."""
        B, T = x.shape; out = 0
        for emb, n in zip(self.hemb, self.ngramas):
            h = torch.zeros(B, T, dtype=torch.long, device=x.device)
            for k in range(n):
                h = h * 257 + (F.pad(x, (k, 0))[:, :T] if k else x) + 1
            out = out + emb(h % self.hsize)
        return out / (len(self.ngramas) + 1)

    def atencion(self, q, k, v, mask):               # q [B,Nq,D]; k,v [B,Nk,D]; mask [B,Nq,Nk] (True = puede atender)
        B, Nq, D = q.shape; Nk = k.shape[1]; H = self.H
        sp = lambda z, N: z.view(B, N, H, D // H).transpose(1, 2)
        o = F.scaled_dot_product_attention(sp(q, Nq), sp(k, Nk), sp(v, Nk), attn_mask=mask.unsqueeze(1))
        return o.transpose(1, 2).reshape(B, Nq, D)

In [ ]:
def forward_blt(self, x, inicios):
    """x [B,T] bytes; inicios [B,T] bool (True = el byte abre un parche; inicios[:,0] = True)."""
    B, T = x.shape; dev = x.device
    pid = inicios.long().cumsum(1) - 1                                   # [B,T] parche de cada byte
    P = int(pid.max()) + 1
    h = self.emb(x) + self.hash_ngramas(x) + self.pos(torch.arange(T, device=dev))
    h = self.enc(h, mask=mascara_causal(T, dev), is_causal=True)         # 1) codificador local (bytes)
    M = pid.unsqueeze(1) == torch.arange(P, device=dev).view(1, P, 1)    # [B,P,T] bytes de cada parche
    pool = h.unsqueeze(1).masked_fill(~M.unsqueeze(-1), -1e4).amax(2)    # consulta inicial = max-pool del parche
    vacio = ~M.any(2); pool = pool.masked_fill(vacio.unsqueeze(-1), 0.)
    M2 = M.clone(); M2[..., 0] |= vacio                                  # (parches de relleno: evita filas sin claves)
    p = self.pool_e(pool) + self.o_e(self.atencion(self.q_e(pool), self.k_e(h), self.v_e(h), M2))   # 2) bytes -> parches
    o = self.glob(p + self.ppos(torch.arange(P, device=dev)), mask=mascara_causal(P, dev), is_causal=True)  # 3) global
    o = torch.cat([self.bos.expand(B, -1, -1), o], 1)                    # o_{-1} = BOS aprendido
    k, v = self.kv_d(o).chunk(2, -1)
    Md = pid.unsqueeze(2) == torch.arange(P + 1, device=dev).view(1, 1, P + 1)   # el byte del parche j atiende a o_{j-1}
    d = h + self.o_d(self.atencion(self.q_d(h), k, v, Md))               # 4) parches -> bytes
    d = self.dec(d, mask=mascara_causal(T, dev), is_causal=True)         # 5) decodificador local
    return self.head(self.ln(d))

MiniBLT.forward = forward_blt

# prueba de causalidad: cambiar el byte t NO debe alterar las predicciones de posiciones < t
m_test = MiniBLT().eval(); xt = torch.randint(0, 256, (2, 64)); st = torch.zeros(2, 64, dtype=torch.bool); st[:, ::5] = True
x2 = xt.clone(); x2[:, 40] = (x2[:, 40] + 1) % 256
with torch.no_grad():
    d_ = (m_test(xt, st) - m_test(x2, st)).abs().amax(-1)
print("máx. diferencia en posiciones < 40 :", float(d_[:, :40].max()), "(debe ser 0)")
print("máx. diferencia en posiciones >= 40:", float(d_[:, 40:].max()), "(debe ser > 0)")

### 9.1 Baselines y entrenamiento comparado

Entrenamos tres modelos con el **mismo transformer global** (mismo ancho y capas), la **misma ventana en bytes** (128, como hace BLT para no dar ventaja de contexto) y el mismo presupuesto de pasos:
(a) transformer **isotrópico sobre bytes**, (b) transformer sobre **tokens BPE** (el baseline de la industria), (c) **MiniBLT** con parches por entropía.

In [ ]:
# --- Datos del baseline BPE: flujo de ids y longitud en bytes de cada token ---
IDS_TR = np.array(TK.encode(TXT_TR).ids, dtype=np.int64)
IDS_VA = np.array(TK.encode(TXT_VA).ids, dtype=np.int64)
TOK_B = np.array([len(TK.id_to_token(i)) for i in range(TK.get_vocab_size())])     # en ByteLevel: 1 carácter = 1 byte
BPT = len(TXT_TR.encode()) / len(IDS_TR)
T_BYTES, T_TOK = 128, int(round(128 / BPT))                                          # misma ventana en bytes
print(f"bytes/token (train) = {BPT:.2f} -> ventana de {T_TOK} tokens ~ {T_BYTES} bytes")

def lote_ids(ids, B, T, g):
    ix = torch.randint(0, len(ids) - T - 1, (B,), generator=g)
    x = torch.stack([torch.from_numpy(ids[i:i + T]) for i in ix]); y = torch.stack([torch.from_numpy(ids[i + 1:i + T + 1]) for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def bpb_bpe(m, n=64):
    g = torch.Generator().manual_seed(1); ix = torch.randint(0, len(IDS_VA) - T_TOK - 1, (n,), generator=g)
    x = torch.stack([torch.from_numpy(IDS_VA[i:i + T_TOK]) for i in ix]).to(device)
    y = torch.stack([torch.from_numpy(IDS_VA[i + 1:i + T_TOK + 1]) for i in ix]).to(device)
    nll = F.cross_entropy(m(x).reshape(-1, m.head.out_features), y.reshape(-1), reduction="sum").item()
    return nll / math.log(2) / TOK_B[y.cpu().numpy()].sum()               # NLL total / bytes totales cubiertos por los targets

In [ ]:
D_G, L_G = (128, 3) if FAST_DEV_RUN else (512, 8)
PASOS = 400 if FAST_DEV_RUN else 4000
B_TR = 16
H_TR_BLT, H_VA_BLT = H_TR, H_VA                                      # entropías (tabla n-grama) precalculadas = como en BLT

def lote_blt(data, Hs, B, T, g):
    ix = torch.randint(0, len(data) - T - 1, (B,), generator=g)
    x = torch.stack([torch.from_numpy(data[i:i + T].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + T + 1].astype(np.int64)) for i in ix])
    st = torch.stack([torch.from_numpy(inicios_global(Hs[i:i + T].copy(), TH_G)) for i in ix])
    return x.to(device), y.to(device), st.to(device)

def entrena(nombre, modelo, paso_fn, eval_fn, pasos=PASOS, lr=2e-3, cada=50):
    opt = torch.optim.AdamW(modelo.parameters(), lr=lr); g = torch.Generator().manual_seed(SEED)
    curva, t0 = [], time.time()
    for s in range(pasos + 1):
        if s % cada == 0:
            modelo.eval(); curva.append((s, eval_fn(modelo))); modelo.train()
        if s == pasos:
            break
        loss = paso_fn(modelo, g); opt.zero_grad(); loss.backward(); opt.step()
    print(f"{nombre:22s} params={sum(p.numel() for p in modelo.parameters())/1e6:5.2f}M | {time.time()-t0:5.0f}s | bpb final = {curva[-1][1]:.3f}")
    return curva

RES = {}
# (a) transformer isotrópico sobre bytes
m_b = TinyGPT(256, D_G, L_G, T_BYTES).to(device)
def paso_b(m, g):
    x, y = lote(TR, B_TR, T_BYTES, g); return F.cross_entropy(m(x).reshape(-1, 256), y.reshape(-1))
def eval_b(m):
    g = torch.Generator().manual_seed(1); x, y = lote(VA, 64, T_BYTES, g)
    with torch.no_grad(): return F.cross_entropy(m(x).reshape(-1, 256), y.reshape(-1)).item() / math.log(2)
RES["bytes"] = entrena("bytes isotrópico", m_b, paso_b, eval_b)
# (b) transformer sobre tokens BPE (mismo tamaño global)
m_t = TinyGPT(TK.get_vocab_size(), D_G, L_G, T_TOK + 1).to(device)
def paso_t(m, g):
    x, y = lote_ids(IDS_TR, B_TR, T_TOK, g); return F.cross_entropy(m(x).reshape(-1, m.head.out_features), y.reshape(-1))
RES["bpe"] = entrena("BPE (tokens)", m_t, paso_t, bpb_bpe)
# (c) MiniBLT con parches por entropía
m_blt = MiniBLT(dl=64 if FAST_DEV_RUN else 256, dg=D_G, Lg=L_G).to(device)
def paso_blt(m, g):
    x, y, st = lote_blt(TR, H_TR_BLT, B_TR, T_BYTES, g); return F.cross_entropy(m(x, st).reshape(-1, 256), y.reshape(-1))
def eval_blt(m):
    g = torch.Generator().manual_seed(1); x, y, st = lote_blt(VA, H_VA_BLT, 64, T_BYTES, g)
    with torch.no_grad(): return F.cross_entropy(m(x, st).reshape(-1, 256), y.reshape(-1)).item() / math.log(2)
RES["blt"] = entrena("MiniBLT (entropía)", m_blt, paso_blt, eval_blt)

In [ ]:
ps_real = len(H_VA) / inicios_global(H_VA, TH_G).sum()
dl_ = 64 if FAST_DEV_RUN else 256
fl = {"bytes": L_G * flops_capa(D_G, T_BYTES) + 2 * D_G * 256,
      "bpe": (L_G * flops_capa(D_G, T_TOK) + 2 * D_G * TK.get_vocab_size()) / BPT,
      "blt": L_G * flops_capa(D_G, T_BYTES / ps_real) / ps_real + (1 + 2) * flops_capa(dl_, T_BYTES) + 2 * dl_ * 256 + 4 * dl_ * D_G}
etq = {"bytes": "bytes isotrópico", "bpe": f"BPE-{VOCABS[1]}", "blt": f"MiniBLT (ps={ps_real:.1f})"}
cols = {"bytes": GRIS, "bpe": NARANJA, "blt": AZUL}
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for k, c in RES.items():
    axs[0].plot([s for s, _ in c], [v for _, v in c], color=cols[k], lw=2, label=etq[k])
axs[0].set_xlabel("paso de entrenamiento"); axs[0].set_ylabel("bits por byte (validación)"); axs[0].set_title("Curvas de validación (escala diminuta)")
axs[0].legend()
for k in RES:
    axs[1].scatter(fl[k] / 1e6, RES[k][-1][1], s=120, color=cols[k], label=etq[k])
axs[1].set_xlabel("MFLOPs por byte (forward, fórmula)"); axs[1].set_ylabel("bpb final (val)"); axs[1].set_title("Calidad vs. cómputo por byte")
axs[1].legend(); plt.tight_layout(); plt.show()
print({etq[k]: f"{fl[k]/1e6:.1f} MFLOPs/B | bpb={RES[k][-1][1]:.3f}" for k in RES})

> **👀 Qué debes observar**
>
> * A esta escala (≈1-2M de parámetros, minutos de CPU, 400 pasos, corpus pequeño o sintético) **BPE aprende mucho más rápido**: en la corrida de referencia del curso (corpus sintético) el transformer sobre BPE-512 llegó a ≈1.0 bpb, MiniBLT a ≈1.7 y el transformer de bytes a ≈2.9. **Es coherente con el paper**: ahí BPE gana con presupuestos pequeños y BLT lo alcanza «no muy lejos del punto compute-óptimo» (cruce en ≈150B-1T bytes según el tamaño, Tabla 2).
> * Lo que sí debes ver: (1) MiniBLT **aprende** y supera al transformer de bytes con ≈ 1/4.3 de los pasos del transformer grande; (2) en FLOPs/byte queda al nivel de BPE; (3) la brecha con BPE depende de los pasos de entrenamiento y de los datos: no la generalices.
> * **No repliques como conclusión** «BLT pierde frente a BPE»: este banco no tiene potencia estadística (una semilla, 300 pasos). Es una prueba de *mecanismo*.

## 10. Resultados publicados: qué dice el paper (y qué no)

Cifras **reportadas por los autores** (Pagnoni et al., arXiv:2412.09871, v1 de diciembre 2024; ACL 2025). **No son nuestras mediciones** y pueden no reproducirse con otra receta.

| Resultado | Cifra reportada | Condiciones |
|---|---|---|
| Escala | Estudio FLOP-controlado hasta 8B parámetros y 4T bytes de entrenamiento | Primer estudio así para modelos de bytes, según los autores |
| Paridad con Llama 3 | Media de 7 benchmarks: 61.1 (BLT-Entropy) vs 60.0 (Llama 3), ambos entrenados con el mismo presupuesto de FLOPs (1T tokens ≈ 4.5T bytes) | 8B, dataset BLT-1T; umbral de inferencia reajustado de 0.6 a 0.1 |
| Ahorro de inferencia | «Hasta 50% menos FLOPs» con parches de 8 bytes (con un coste menor en métricas) | Escalado con FLOPs de inferencia fijos: baselines de ≈ 470M y 3.6B (tokenizer de Llama 2) y 450M y 3.9B (Llama 3) |
| Robustez al ruido | HellaSwag con ruido (media): 64.3 vs 56.9 (Llama 3, 1T tokens) y 64.3 (Llama 3.1, 16T tokens) | 8B |
| Conciencia ortográfica | CUTE: 54.1 vs 27.5 (Llama 3) y 20.0 (Llama 3.1); «contains char» 55.9 vs 0.0 | 8B |
| Traducción multilingüe | FLORES-101, media de 27 idiomas X→En: 14.0 vs 12.1; En→X: 6.4 vs 5.9 | 8B, 1T tokens |

In [ ]:
# Cifras REPORTADAS por Pagnoni et al. (2024), arXiv:2412.09871 (tablas 1, 3 y 4). NO las hemos reproducido.
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
n1, v1 = ["Llama 3\n(BPE, 1T tok)", "BLT-Space\n(6.1 B/parche)", "BLT-Entropy\n(4.5 B/parche)"], [60.0, 58.0, 61.1]
axs[0].bar(n1, v1, color=[NARANJA, GRIS, AZUL]); axs[0].set_ylim(50, 64); axs[0].set_title("Media de 7 benchmarks, 8B (Tabla 1)", fontsize=9)
for i, v in enumerate(v1): axs[0].text(i, v + .2, f"{v}", ha="center")
n2, v2 = ["Llama 3\n(1T tok)", "Llama 3.1\n(16T tok)", "BLT\n(1T tok)"], [27.5, 20.0, 54.1]
axs[1].bar(n2, v2, color=[NARANJA, "#d9b38c", AZUL]); axs[1].set_title("CUTE (manipulación de caracteres), 8B (Tabla 3)", fontsize=9)
for i, v in enumerate(v2): axs[1].text(i, v + .8, f"{v}", ha="center")
n3 = ["Ruido en\nHellaSwag (media)", "FLORES-101\nX→En (media)", "FLORES-101\nEn→X (media)"]
l3, b3 = [56.9, 12.1, 5.9], [64.3, 14.0, 6.4]
x = np.arange(3); axs[2].bar(x - .2, l3, .4, color=NARANJA, label="Llama 3"); axs[2].bar(x + .2, b3, .4, color=AZUL, label="BLT")
axs[2].set_xticks(x); axs[2].set_xticklabels(n3, fontsize=8); axs[2].legend(); axs[2].set_title("Robustez y traducción (Tablas 3-4)", fontsize=9)
plt.tight_layout(); plt.show()

> **👀 Qué debes observar**
>
> * La ganancia más llamativa **no es el promedio de benchmarks de conocimiento** (diferencia de ~1 punto) sino **las tareas de caracteres**: CUTE pasa de 27.5 a 54.1, y Llama 3.1 con 16x más datos *no* lo arregla (20.0). La conciencia ortográfica **no se compra solo con datos** si el modelo no ve bytes.
> * **Qué NO demuestran**: que BLT supere a BPE en *cualquier* escala (el paper no hace leyes de escala propias, §9); ni que sea más rápido en tiempo de reloj; ni que valga para codificadores de embeddings (BLT es un decodificador de lenguaje).

> ### 🎬 GEMINI-VIDEO · ID: `M06-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Patching dinámico por entropía, paso a paso**
>
> **Qué muestra:** Animación del proceso: llegan bytes, el modelo de entropía mide la incertidumbre, se supera el umbral y se abre un parche; el transformer grande solo se activa al cerrar cada parche.
> **Formato:** MP4 16:9, 1080p | Duración: 25 s | **Texto alternativo:** Animación en la que una fila de bytes se agrupa en parches cuando la entropía supera un umbral y el transformer grande se activa una vez por parche
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Video explicativo técnico de 25 segundos, 16:9, 1080p, ritmo calmado, sin voz en off, sin música intensa (opcional un fondo muy suave). Estilo: diagrama limpio sobre fondo blanco, tipografía sans-serif legible, paleta azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Sin ilustraciones decorativas.
> Texto de la escena: la frase "El servidor procesa las peticiones." escrita como una fila de cajitas, una por byte (espacios como ␣), en el centro-inferior de la pantalla.
> Guion por escenas:
> 0–4 s: Título "Parches dinámicos por entropía" aparece y se desvanece. Aparece la fila de cajitas grises vacías y, sobre ella, un eje con la etiqueta "H(siguiente byte) en bits" y una línea roja discontinua horizontal etiquetada "umbral θ".
> 4–14 s: los bytes se iluminan de izquierda a derecha, uno cada ~0.3 s. Sobre cada byte crece una barra azul con la entropía: alta (supera θ) en "E", "s" (de "servidor"), "p" (de "procesa"), "l" (de "las"), "p" (de "peticiones"); baja en el resto de letras dentro de cada palabra. Cuando una barra supera la línea roja se vuelve naranja y aparece una línea vertical verde que abre un nuevo parche; el parche anterior se rellena de verde claro y se cierra con un pequeño "clic" visual.
> 14–20 s: los parches cerrados suben hacia un bloque grande etiquetado "Transformer latente global" que parpadea UNA vez por parche; debajo, contador "pasos del transformer grande: 5" frente a un contador gris "bytes: 35".
> 20–25 s: texto final en pantalla: "Mismo contenido: 35 bytes → 5 parches. El umbral θ controla el coste." Se desvanece.
> Texto en pantalla solo en español, con términos técnicos en inglés (patch, threshold).
> Evita: voz en off, texto ilegible, cambios bruscos de cámara, números distintos de los indicados, logos, decoraciones.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M06-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M06-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> t=0s   [bytes vacíos] ---- línea θ ---------------------
> t=4-14s  E l ␣ s e r v i d o r ...   barras de entropía: ▂▇ ▂ ▇▂▂▂▂▂ ...  (naranja = supera θ -> | nuevo parche)
> t=14-20s parches cerrados -> [Transformer global] parpadea 1 vez por parche   (5 pasos vs 35 bytes)
> t=20-25s "35 bytes -> 5 parches. θ controla el coste."
> ```

> **👀 Qué debes observar**
>
> Fíjate en el orden causal: el umbral se evalúa **con el pasado** (no hay que mirar adelante), el parche se **cierra** cuando aparece el siguiente byte difícil y solo entonces corre el bloque caro. Es lo que permite generar byte a byte con parches dinámicos.

## 11. Con librerías de industria: ByT5, CANINE y BLT en Hugging Face

In [ ]:
from transformers import ByT5Tokenizer, CanineTokenizer
bt5, can = ByT5Tokenizer(), CanineTokenizer()          # ambos funcionan SIN descargar nada: no hay vocabulario que aprender
frases = {"es": "Canción de cuna para el ñandú",
          "en": "A lullaby for the rhea",
          "code": "def f(x): return x ** 2  # 🙂"}
print(f"{'texto':32s} {'chars':>5s} {'bytes':>5s} {'ByT5':>5s} {'CANINE':>6s} {'BPE-'+str(VOCABS[1]):>8s}")
for l, t in frases.items():
    print(f"{t[:30]:32s} {len(t):5d} {len(t.encode()):5d} {len(bt5(t).input_ids)-1:5d} {len(can(t).input_ids)-2:6d} {len(TK.encode(t).ids):8d}")
print("ByT5 ids de 'ñ':", bt5("ñ").input_ids[:-1], "= bytes UTF-8", list("ñ".encode()), "+ 3 (ids 0-2 reservados: pad/eos/unk)")
print("CANINE ids de 'ñ🙂':", can("ñ🙂").input_ids[1:-1], "= code points", [ord(c) for c in "ñ🙂"])
try:                                                    # BLT oficial (requiere aceptar la licencia en HF y bajar ~GB)
    from transformers import BltForCausalLM
    print("transformers incluye BltForCausalLM; los pesos oficiales están en el hub (p. ej. facebook/blt-1b, con acceso aprobado).")
except Exception as e:
    print("Tu versión de transformers no incluye BLT:", type(e).__name__, "-> usa el repo facebookresearch/blt")

> **👀 Qué debes observar**
>
> * **ByT5** no tiene vocabulario que aprender: ids = bytes + 3 reservados; **CANINE** usa directamente *code points* de Unicode (por eso un emoji es 1 posición, pero 4 bytes). Las longitudes muestran el coste: ~1 posición por byte (ByT5) o por carácter (CANINE) frente a ~4 bytes/token de BPE.
> * Para BLT usa el repositorio `facebookresearch/blt` o los pesos del hub; la clase `BltForCausalLM` de `transformers` existe en versiones recientes, pero necesita pesos convertidos: **verifica el nombre del checkpoint** antes de planificar un experimento (los pesos oficiales de Meta son de acceso controlado).

## 12. Mapa de madurez (a oct-2026)

Tras ver el mecanismo, el mini-experimento y lo que dice el paper, esta tabla resume qué es sólido y qué no.

| Técnica | Madurez | Quién la usa / respaldo público | Fuente |
|---|---|---|---|
| Subword BPE (referencia) | 🟢 | Prácticamente todos los LLM abiertos con tokenizer documentado | Módulos 02-04 |
| CANINE (encoder a nivel de carácter con *downsampling*) | 🟡 | Google Research; pesos publicados; no es base de modelos de frontera | Clark et al., TACL 2022 (arXiv:2103.06874) |
| ByT5 (T5 byte-a-byte) | 🟡 | Google Research; robusto a ruido, 4x más largo | Xue et al., TACL 2022 (arXiv:2105.13626) |
| MegaByte (parches fijos, local+global) | 🔴 | Meta; el propio paper de BLT lo ve por detrás de BPE a escala | Yu et al., 2023 (arXiv:2305.07185) |
| SpaceByte (parche tras cada espacio) | 🔴 | Investigación (autor único, NeurIPS 2024); a cómputo fijo iguala aproximadamente a los Transformers tokenizados en sus experimentos | Slagle, 2024 (arXiv:2404.14408) |
| MambaByte (SSM sobre bytes) | 🔴 | Investigación (COLM 2024); modelos de 353M a 972M (y 1.6B en una prueba de velocidad) | Wang et al., 2024 (arXiv:2401.13660) |
| **BLT: parches por entropía** | 🟡/🔴 | Meta FAIR; hasta 8B / 4T bytes; ACL 2025; pesos 1B y 7B en HF (acceso controlado) | Pagnoni et al., 2024 (arXiv:2412.09871) |
| *Byte-ification* de un LLM ya entrenado (Bolmo) | 🔴 | Ai2: Bolmo 1B/7B a partir de Olmo 2 1B / Olmo 3 7B | Minixhofer et al., 2025 (arXiv:2512.15586) |
| Chunking aprendido de extremo a extremo (H-Net) | 🔴 | Cartesia/CMU (módulo 07) | Hwang, Wang, Gu (arXiv:2507.07955) |

> **Honestidad.** «🟡/🔴» significa: *hay un resultado a escala que respalda la idea, pero no hay evidencia pública de un LLM de frontera en producción con ella*. «Evidencia pública» ≠ «nadie lo usa»:
> los laboratorios no siempre documentan su tokenizer. Lo que sí es verificable es lo que aparece en los papers y en los repositorios abiertos.

## 13. 🏭 En producción

* **Quién lo hace público.** Meta FAIR publicó código y pesos de BLT (1B y 7B en el hub de Hugging Face; el README del repositorio remite a una licencia no comercial: léela antes de usarlo). Ai2 publicó **Bolmo 1B/7B**, obtenidos por *byte-ification* de Olmo 2 1B / Olmo 3 7B, y reporta que la conversión usa una fracción pequeña del presupuesto de preentrenamiento (arXiv:2512.15586). Cartesia/CMU publicaron los pesos de H-Net (módulo 07).
* **Lo que NO está documentado.** No he encontrado evidencia pública de un LLM de frontera cerrado (OpenAI, Anthropic, Google, Mistral…) que use parches de bytes en producción; sus tokenizers, cuando se documentan, siguen siendo de subpalabras. Trátalo como «sin evidencia pública», no como «no existe».
* **Fricciones de ingeniería** (del propio paper y de H-Net): las librerías están optimizadas para tokens fijos; la atención cruzada con máscaras por parche necesita `FlexAttention` o kernels propios; el cómputo y la memoria son **dinámicos** (lotes con parches desiguales) y el *batching* de inferencia con pasos de coste distinto es difícil; ahorro en FLOPs ≠ ahorro en milisegundos.
* **Dónde sí compensa hoy** (hipótesis razonables, no garantías): dominios donde el vocabulario no existe o es mal ajustado (ADN, código con símbolos raros, idiomas poco representados), tareas sensibles a la ortografía y entradas con ruido. Para **embedders de Políglota** (módulos 09-12), el estándar sigue siendo un encoder con tokenizer: BLT es un decodificador y no hay un recetario de embedders por parches.
* **Coste de cambiar de paradigma:** pierdes el ecosistema (caché de prefijos por token, *constrained decoding* por vocabulario, *logit bias*, facturación por token, evaluación por token) o tienes que rehacerlo.

## 14. 🧠 Secretos de la élite

1. **bpb o nada.** Comparar pérdida por token entre tokenizers es un error de categoría; compara **bits por byte** (o FLOPs por byte a calidad igual). Y recuerda que, para modelos tokenizados, bpb asume la tokenización canónica (H-Net, §Discusión).
2. **Iguala el contexto en bytes.** BLT mantiene constante el nº de bytes por lote (16M) y por secuencia (8k, luego 16k bytes) para que un patch size mayor no «trampee» con contexto más largo (§4.3). En comparativas bytes-vs-tokens, el control más olvidado es la ventana *en bytes*.
3. **El modelo de entropía es chico y barato… pero sensible a su contexto.** En BLT, el patching empieza a producir parches *progresivamente más largos* en contenido repetitivo (opciones de MMLU) por «entropy drift»; lo mitigan reiniciando el contexto del modelo de entropía en saltos de línea y usando la restricción monótona (§4.4).
4. **El umbral es una perilla de inferencia.** BLT-Entropy se entrena con un umbral de 0.6 y en evaluación lo bajan a 0.1, ganando precisión a costa de más pasos (§5.2): el *patch size* es una palanca de calidad/coste que BPE no ofrece.
5. **Los hash de n-gramas son parte del resultado, no un adorno.** La ablación (Tabla 8) los llama «vitales» para igualar a los tokenizados; sin ellos el codificador local debería ser mucho mayor.
6. **Más datos no arreglan lo que el tokenizer oculta.** CUTE: Llama 3.1 (16T tokens) obtiene 20.0 frente a 54.1 de BLT (1T) (Tabla 3). Si tu producto depende de manipulación de caracteres, el cuello de botella es la representación.
7. **Las leyes de escala heredadas pueden ser subóptimas.** BLT usa la proporción datos/parámetros compute-óptima de Llama 3 (hecha para BPE); los autores admiten que podría no ser la óptima para BLT (§9).
8. **El FLOP no es el reloj.** El paper reconoce que su implementación puede no igualar el tiempo de reloj de los transformers tokenizados (§9). Mide latencia p50/p99 reales antes de vender el ahorro.

## 15. ⚠️ Errores comunes

* **Usar perplejidad por token** para comparar byte-level con BPE. Usa bpb.
* **Dar más contexto en bytes** al modelo de parches que al baseline y atribuirle la mejora al patching.
* **Calibrar el umbral en validación o test** (o con otra mezcla de datos): calíbralo en entrenamiento y reporta el tamaño medio de parche *resultante* en test. Cambiar de dominio cambia el tamaño de parche (y el coste).
* **Dejar fuga de futuro:** si el parche abre en $i$ por un cálculo que mira $x_{\ge i}$, el modelo ve el futuro. Verifica causalidad con una prueba de perturbación (como la de MiniBLT).
* **Creer que «tokenizer-free» = «sin preprocesado»:** sigues necesitando normalización Unicode (módulo 01), límites de longitud y manejo de bytes inválidos.
* **Generalizar un banco diminuto:** 300 pasos y una semilla no concluyen nada sobre 8B. Repite con semillas, aumenta datos y reporta incertidumbre antes de afirmar.

## 16. Autoevaluación

<details>
<summary>1 · (Cálculo) Un BPE da 4.4 bytes/token y BLT 8 bytes/parche. Con el mismo modelo global, ¿qué fracción de los pasos del transformer grande se ahorra? ¿Y de los FLOPs totales si el modelo global es el 100% del coste de BPE y los modelos locales de BLT añaden un 10% de ese coste?</summary>

Pasos globales por byte: 1/4.4 = 0.227 frente a 1/8 = 0.125, es decir 0.125/0.227 = 0.55 → **−45%** de pasos del transformer grande. FLOPs totales de BLT relativos a BPE: 0.55 (global) + 0.10 (locales) = **0.65**, es decir ≈ −35%. Cuanto más dominante sea el global, más cerca de ese −45%; el paper habla de ahorros «de hasta ~50%» con parches de 8 bytes.

</details>

<details>
<summary>2 · (Cálculo) Un modelo BPE tiene entropía cruzada de 2.1 nats por token y 4.4 bytes/token. ¿Cuál es su bpb?</summary>

bpb = 2.1 / (ln 2 · 4.4) = 2.1 / 3.050 = **0.689** bits por byte.

</details>

<details>
<summary>3 · (Diagnóstico) Tu modelo de parches produce parches cada vez más largos al final de un prompt con muchas opciones A/B/C/D repetidas. ¿Qué sucede y cómo se mitiga?</summary>

El modelo de entropía ve el patrón repetido en su contexto y su entropía cae (*entropy drift*), con lo que casi nunca supera el umbral. BLT lo mitiga reiniciando el contexto del modelo de entropía en saltos de línea y usando la restricción monótona aproximada en vez del umbral global.

</details>

<details>
<summary>4 · (Diagnóstico) Una compañera mide «BLT-diminuto pierde con BPE-diminuto por 0.05 bpb» con una semilla y 300 pasos. ¿Qué objetarías?</summary>

Falta de potencia: una semilla, pocos pasos, corpus pequeño/sintético, modelos fuera del régimen compute-óptimo (el paper ve que BPE gana con poco cómputo y BLT cruza más tarde). Hay que repetir con semillas, reportar intervalos y comparar a FLOPs iguales.

</details>

<details>
<summary>5 · (Transferencia) Políglota quiere un embedder multilingüe ES/EN/código robusto a erratas. ¿Qué idea de este módulo reutilizarías y cuál no?</summary>

Reutilizable: agrupar por entropía/costes variables, hash de n-gramas como entrada subléxica robusta (fastText-style), evaluación con perturbaciones de ortografía. No reutilizable tal cual: BLT es un decodificador de lenguaje y su salida son bytes; para un embedder necesitas pooling sobre una representación de secuencia (CANINE es un encoder, pero no es SOTA en retrieval).

</details>

<details>
<summary>6 · (Transferencia) ¿Por qué un generador autoregresivo con BPE necesita un tokenizer externo y BLT no necesita un segmentador externo en inferencia?</summary>

BPE no es incremental: la segmentación de un prefijo depende del futuro, así que se aplica *antes*. El parcheo por entropía es incremental: la decisión en el byte $i$ solo depende de $x_{<i}$, por lo que se puede ejecutar durante la generación.

</details>

<details>
<summary>7 · (Recuerdo) ¿Por qué BLT añade embeddings hash de n-gramas al codificador?</summary>

Dan al codificador información de n-gramas frecuentes (3…8) sin un vocabulario de tokens, lo que mejora mucho el bpb (Tabla 8) y permite un codificador local de una sola capa.

</details>

## 17. Referencias

Todas verificadas (título/autores/fecha) consultando arXiv o el ACL Anthology en la fecha de redacción, y de nuevo (venue, licencia, clases de `transformers` 5.20) el 26-oct-2026.

* Pagnoni, Pasunuru, Rodriguez, Nguyen, Muller, Li, Zhou, Yu, Weston, Zettlemoyer, Ghosh, Lewis, Holtzman, Iyer. **Byte Latent Transformer: Patches Scale Better Than Tokens.** arXiv:2412.09871 (dic. 2024); ACL 2025 (Long Papers), <https://aclanthology.org/2025.acl-long.453>. Código: <https://github.com/facebookresearch/blt> (licencia CC-BY-NC-4.0; pesos 1B y 7B en HF con solicitud de acceso).
* Clark, Garrette, Turc, Wieting. **CANINE: Pre-training an Efficient Tokenization-Free Encoder for Language Representation.** TACL 2022, arXiv:2103.06874.
* Xue et al. **ByT5: Towards a token-free future with pre-trained byte-to-byte models.** TACL 2022, arXiv:2105.13626.
* Yu, Simig, Flaherty, Aghajanyan, Zettlemoyer, Lewis. **MEGABYTE: Predicting Million-byte Sequences with Multiscale Transformers.** NeurIPS 2023, arXiv:2305.07185.
* Slagle. **SpaceByte: Towards Deleting Tokenization from Large Language Modeling.** NeurIPS 2024, arXiv:2404.14408.
* Wang, Gangavarapu, Yan, Rush. **MambaByte: Token-free Selective State Space Model.** COLM 2024, arXiv:2401.13660.
* Nawrot et al. **Hierarchical Transformers Are More Efficient Language Models.** Findings of NAACL 2022, arXiv:2110.13711. · Nawrot et al. **Efficient Transformers with Dynamic Token Pooling.** ACL 2023, arXiv:2211.09761.
* Hwang, Wang, Gu. **Dynamic Chunking for End-to-End Hierarchical Sequence Modeling.** arXiv:2507.07955 (ICLR 2026) — módulo 07.
* Videau et al. **From Bytes to Ideas: Language Modeling with Autoregressive U-Nets.** NeurIPS 2025, arXiv:2506.14761.
* Minixhofer et al. **Bolmo: Byteifying the Next Generation of Language Models.** arXiv:2512.15586.
* Schmidt et al. **Tokenization Is More Than Compression.** EMNLP 2024, arXiv:2402.18376.
* Neitemeier et al. **Hierarchical Autoregressive Transformers: Combining Byte- and Word-Level Processing for Robust, Adaptable Language Models.** arXiv:2501.10322 (preprint).
* Documentación: Hugging Face `transformers` (`ByT5Tokenizer`, `CanineTokenizer`, `BltForCausalLM`).

## 18. Siguiente paso

El **proyecto** (`06_proyecto_tokenizer_free_bytes_blt.ipynb`) te pide construir un *patcher entrópico* y compararlo con BPE en estabilidad, compresión y coste. En el **módulo 07** el segmentador deja de ser una heurística externa y se **aprende** (H-Net), y veremos cómo *cambiar el tokenizer* de un modelo ya entrenado.